## SUNT OD Data Loader
Downloads, validates, and download the SUNT origin-destination dataset

In [ ]:
# --- setup: install dependencies and mount google drive ---
!pip install -q pandas pyarrow huggingface_hub tqdm

from google.colab import drive
drive.mount('/content/drive')

import os

# path to your cloned repo inside drive
REPO_PATH = "/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone"
%cd {REPO_PATH}

# git identity
!git config --global user.email "eleonorvilla2003@email.com"
!git config --global user.name "victoriaeleonor"

# pull latest changes from github
!git pull

In [ ]:
# --- imports ---
from huggingface_hub import hf_hub_download
import pandas as pd
from datetime import datetime
import calendar
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- configuration ---

# months to download: list of (year, month) tuples
# example: download march and april 2024
# MONTHS_TO_DOWNLOAD = [(2024, 3), (2024, 4)]
# to download just one month:
MONTHS_TO_DOWNLOAD = [(2024, 3)]

# where to save the processed data inside your drive
SAVE_PATH = "/content/drive/MyDrive/Occupancy_capstone/Dataset"

# standard bus capacity assumed for occupancy calculation
BUS_CAPACITY = 80

# show download progress bar
SHOW_PROGRESS = True

# validate data after downloading
VALIDATE_DATA = True

In [ ]:
# --- download one month from hugging face ---

def download_month(year, month):
    """
    downloads all available days for a given month from the sunt od dataset.
    returns a concatenated dataframe, or none if nothing could be downloaded.
    """
    num_days = calendar.monthrange(year, month)[1]
    month_name = calendar.month_name[month]

    print(f"\n{'='*60}")
    print(f"downloading sunt od - {month_name} {year}")
    print(f"{'='*60}")
    print(f"days in month: {num_days}")

    dfs = []
    failed_days = []

    day_range = tqdm(range(1, num_days + 1), desc="downloading", unit="day") if SHOW_PROGRESS else range(1, num_days + 1)

    for day in day_range:
        date_str = datetime(year, month, day).strftime("%Y-%m-%d")
        filename = f"OD/od-{date_str}.parquet"

        try:
            file_path = hf_hub_download(
                repo_id="labiaufba/PublicTransportationSunt",
                filename=filename,
                repo_type="dataset"
            )
            df_day = pd.read_parquet(file_path)

            if len(df_day) == 0:
                failed_days.append(date_str)
                continue

            dfs.append(df_day)

        except Exception:
            failed_days.append(date_str)
            continue

    print(f"\ndays downloaded successfully: {num_days - len(failed_days)}/{num_days}")

    if not dfs:
        print("error: could not download any day for this month")
        return None

    df = pd.concat(dfs, ignore_index=True)
    print(f"total records: {len(df):,}")
    print(f"memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} mb")

    return df

In [ ]:
# --- validate downloaded data ---

def validate_data(df):
    """
    checks that all critical columns are present and shows basic stats.
    returns true if data is valid, false otherwise.
    """
    print(f"\n{'='*60}")
    print("data validation")
    print(f"{'='*60}")

    # columns we need for the occupancy model
    critical_columns = {
        'loading':          'passenger count on bus (critical)',
        'n_boardings':      'number of boardings at stop',
        'n_alighting':      'number of alightings at stop',
        'route_short_name': 'bus line',
        'stop_id':          'stop identifier',
        'direction_id':     'trip direction',
        'pt_sequence':      'stop sequence within trip',
        'gps_datetime':     'timestamp',
    }

    missing = [col for col in critical_columns if col not in df.columns]

    for col, desc in critical_columns.items():
        status = "ok" if col in df.columns else "MISSING"
        print(f"  {status:8s} {col:20s} - {desc}")

    if missing:
        print(f"\nwarning: {len(missing)} critical columns are missing")
        return False

    # loading stats
    print(f"\nloading (passenger count) stats:")
    print(f"  valid values : {df['loading'].notna().sum():,}")
    print(f"  nan values   : {df['loading'].isna().sum():,} ({df['loading'].isna().mean()*100:.2f}%)")
    print(f"  mean         : {df['loading'].mean():.2f} passengers")
    print(f"  max          : {df['loading'].max():.0f} passengers")
    print(f"  negative     : {(df['loading'] < 0).sum():,} records (will be filtered)")

    # date range
    if 'gps_datetime' in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df['gps_datetime']):
            df['gps_datetime'] = pd.to_datetime(df['gps_datetime'], errors='coerce')
        print(f"\ndate range: {df['gps_datetime'].min()} → {df['gps_datetime'].max()}")

    print(f"\nvalidation complete")
    return True